# Day 40: Polynomial Regression

## Why Polynomial Regression?
Real-world relationships are often curved, not straight - think of how house price grows with size, or how a car's stopping distance grows with speed. A plain straight-line (linear) model can only draw a straight line through the data, so it underfits any curved pattern. Polynomial regression fixes this by adding powers of the input (x^2, x^3, ...) as extra features, letting a linear model bend to follow curves while still being solved with the same simple linear regression math. The cell below just creates and visualizes some synthetic curved data (a parabola with noise) so we have something to fit in the next steps. When you look at the scatter plot, notice the U-shape - that's the kind of pattern a straight line would fail to capture.</cell id="cell-1">

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)  # fix randomness so results are reproducible
X = np.linspace(-3, 3, 40).reshape(-1, 1)  # 40 evenly spaced x values, shaped as a column for sklearn
y = X.ravel() ** 2 + np.random.normal(0, 1, 40)  # true relationship is quadratic (x^2) plus random noise
plt.scatter(X, y)
plt.title('Curved Data')
plt.show()

## Transforming Features
`PolynomialFeatures` is the tool that turns a single feature x into multiple features (x, x^2, and higher powers depending on `degree`). This matters because `LinearRegression` itself only ever fits straight lines/planes - it has no idea what "curved" means. By feeding it x^2 as an extra column, we let it find a linear combination of x and x^2 that traces a curve in the original x-y space, even though the model is still technically linear in its parameters. Here we use `degree=2`, which is enough to capture the parabola shape from the previous cell. After fitting, `model` holds the learned coefficients for the intercept, x, and x^2 terms.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly = PolynomialFeatures(degree=2)  # will generate [1, x, x^2] columns
X_poly = poly.fit_transform(X)  # transform original X into the expanded feature set
model = LinearRegression().fit(X_poly, y)  # ordinary linear regression, but on the expanded features

## Comparing Fits
It helps to see underfitting side by side with a better fit. Here we train a plain linear regression on the raw X (no polynomial features) and plot it next to the degree-2 polynomial model from before. The linear model is forced to draw a single straight line through clearly curved data, so it systematically misses the low points near x=0 and the high points at the edges - a classic sign of underfitting (high bias). The polynomial curve, by contrast, follows the U-shape of the data closely. When judging a fit visually like this, look for whether the line/curve tracks the general trend of the scatter points without chasing every noisy wiggle.

In [ ]:
linear_model = LinearRegression().fit(X, y)  # baseline: straight-line fit on raw (untransformed) X
X_sorted = np.sort(X, axis=0)  # sort so the line/curve plots smoothly left to right
plt.scatter(X, y, alpha=0.5)
plt.plot(X_sorted, linear_model.predict(X_sorted), label='Linear', color='red')
plt.plot(X_sorted, model.predict(poly.transform(X_sorted)), label='Polynomial (deg 2)', color='green')  # apply the same poly transform before predicting
plt.legend()
plt.show()

## Danger of Overfitting
More degrees give the model more flexibility to bend - but past a point it stops learning the true underlying pattern and starts memorizing the random noise in the training data. This is overfitting (high variance): the curve wiggles wildly to pass close to every individual point, which looks like a great fit on this exact dataset but generalizes terribly to new data. Here we crank the degree up to 15 to deliberately produce this effect. When you look at the plot, the giveaway is erratic, wavy behavior between points (especially near the edges) instead of a smooth trend - in practice you'd catch this not by eyeballing a plot but by seeing training error stay low while validation/test error rises, which is why picking the right degree always requires checking performance on held-out data.

In [ ]:
poly_high = PolynomialFeatures(degree=15)  # much higher degree -> much more flexible (and overfit-prone) model
X_poly_high = poly_high.fit_transform(X)
model_high = LinearRegression().fit(X_poly_high, y)
plt.scatter(X, y, alpha=0.5)
plt.plot(X_sorted, model_high.predict(poly_high.transform(X_sorted)), color='purple', label='Degree 15 (overfit)')  # note the wild wiggles vs. the smooth degree-2 curve
plt.legend()
plt.show()